In [1]:


import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt 
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from torchvision.transforms import Compose
import cv2
from ultralytics import YOLO
import random
from itertools import chain
import csv

     


In [2]:
model = YOLO("yolo11m.pt")

image_dir = Path("dataset/singleObjectDetection/test/Boat2/input")

image_paths = sorted(image_dir.glob("*.jpg"))

print(f"Found {len(image_paths)} images")


positions = []

for frame_idx, image_path in enumerate(image_paths):

    image = cv2.imread(str(image_path))

    result = model(image,classes=[8],conf=0.2,iou=0.05, verbose=False)[0]

    for box in result.boxes:

        class_id = int(box.cls.item())
        confidence = float(box.conf.item())

        x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()

        cx = (x1 + x2) / 2
        cy = y2-20

        positions.append({
            "frame": frame_idx,
            "filename": image_path.name,
            "confidence": confidence,
            "x1": x1,
            "y1": y1,
            "x2": x2,
            "y2": y2,
            "cx": cx,
            "cy": cy
        })



setPos = set()
for dict in positions:
    if(dict["frame"] not in setPos):
        setPos.add(dict["frame"])
    else:
        print(dict["frame"])

detected_frames = {position["frame"] for position in positions}
expected_frames = set(range(len(image_paths)))

for frame in sorted(expected_frames - detected_frames):
    print("missing detection in frame", frame)

df = pd.DataFrame(positions)

df.to_csv("positions.csv", index=False)

df.head()

Found 0 images


""


In [3]:
result = model(str(image_paths[430]),classes=[8],conf=0.2,iou=0.05)[0]

print(result.boxes)

annotated = result.plot()

plt.figure(figsize=(10, 6))
plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.show()

IndexError: list index out of range

In [4]:

def generate_angles(num_frames, max_angle=50, max_step=10, seed=None):
    rng = np.random.default_rng(seed)
    angles = np.zeros(num_frames, dtype=np.float32)
    velocity = 0.0

    for t in range(1, num_frames):
        velocity = np.clip(
            0.85 * velocity + rng.normal(0, 0.4 * max_step),
            -max_step,
            max_step,
        )
        angles[t] = np.clip(angles[t - 1] + velocity, -max_angle, max_angle)

    return angles


def calculate_zoom(w, h, max_angle=50):
    angles = np.linspace(-max_angle, max_angle, 181)
    zoom_factors = []

    for angle in angles:
        theta = np.deg2rad(angle)
        c = abs(np.cos(theta))
        s = abs(np.sin(theta))
        zoom_x = c + (h / w) * s
        zoom_y = c + (w / h) * s
        zoom_factors.append(max(zoom_x, zoom_y))

    return max(zoom_factors) * 1.005


# Paths to folders containing images
image_dirs = [
    Path('dataset/singleObjectDetection/train/Person1/input'),
    Path('dataset/singleObjectDetection/train/Person3/input'),
    Path('dataset/singleObjectDetection/train/Person4/input'),
    Path('dataset/singleObjectDetection/train/sofaMan1/input'),
    Path('dataset/singleObjectDetection/train/sofaMan2/input'),
    Path('dataset/singleObjectDetection/train/sofaMan4/input'),
    Path('dataset/singleObjectDetection/val/Person2/input'),
    Path('dataset/singleObjectDetection/val/sofaMan3/input'),
    Path('dataset/singleObjectDetection/test/Boat1/input'),
    Path('dataset/singleObjectDetection/test/Boat2/input'),
]

num_trajectories = 200
max_angle = 50
max_step = 10

for video_idx, image_dir in enumerate(image_dirs):
    filenames = sorted(p.name for p in image_dir.iterdir() if p.suffix.lower() in ('.jpg', '.jpeg', '.png'))
    h, w = cv2.imread(str(image_dir / filenames[0])).shape[:2]
    zoom = calculate_zoom(w, h, max_angle)

    output_dir = image_dir.parent / 'rotation_labels'
    output_dir.mkdir(exist_ok=True)

    for trajectory_idx in range(num_trajectories):
        angles = generate_angles(
            len(filenames),
            max_angle=max_angle,
            max_step=max_step,
            seed=video_idx * num_trajectories + trajectory_idx,
        )

        with (output_dir / f'rotation_{trajectory_idx:03d}.csv').open('w', newline='') as f:
            writer = csv.writer(f)
            writer.writerow(['filename', 'angle_deg', 'zoom'])
            writer.writerows(zip(filenames, angles, [zoom] * len(filenames)))

FileNotFoundError: [Errno 2] No such file or directory: 'dataset/singleObjectDetection/train/Person1/input'